In [1]:
import pandas as pd
from google.colab import files

uploaded = files.upload()

Saving Online Retail.xlsx to Online Retail.xlsx


In [2]:
df = pd.read_excel("Online Retail.xlsx")
df = df.drop_duplicates().copy()

df["InvoiceDate"] = pd.to_datetime(
    df["InvoiceDate"], errors="coerce"
)

df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [3]:
# Keep valid positive-price, positive-quantity transactions
dml_data = df[
    (df["UnitPrice"] > 0) &
    (df["Quantity"] > 0) &
    (df["InvoiceDate"].notna())
].copy()

# Treatment and outcome
dml_data["T"] = dml_data["UnitPrice"]
dml_data["Y"] = dml_data["Quantity"]

# Time-based covariates
dml_data["InvoiceMonth"] = dml_data["InvoiceDate"].dt.month
dml_data["InvoiceHour"] = dml_data["InvoiceDate"].dt.hour
dml_data["DayOfWeek"] = dml_data["InvoiceDate"].dt.dayofweek

print("Original cleaned rows:", len(df))
print("Rows for positive-price/quantity analysis:", len(dml_data))
print("Treatment (T): UnitPrice")
print("Outcome (Y): Quantity")
print("Covariates:", ["InvoiceMonth", "InvoiceHour", "DayOfWeek"])

dml_data[["T", "Y", "InvoiceMonth", "InvoiceHour", "DayOfWeek"]].head()

Original cleaned rows: 536641
Rows for positive-price/quantity analysis: 524878
Treatment (T): UnitPrice
Outcome (Y): Quantity
Covariates: ['InvoiceMonth', 'InvoiceHour', 'DayOfWeek']


,T,Y,InvoiceMonth,InvoiceHour,DayOfWeek
0,2.55,6,12,8,2
1,3.39,6,12,8,2
2,2.75,8,12,8,2
3,3.39,6,12,8,2
4,3.39,6,12,8,2


In [4]:
summary = pd.DataFrame({
    "Metric": [
        "Rows before filtering",
        "Rows after filtering",
        "Treatment variable",
        "Outcome variable",
        "Number of covariates",
        "Missing treatment values",
        "Missing outcome values"
    ],
    "Value": [
        len(df),
        len(dml_data),
        "UnitPrice",
        "Quantity",
        3,
        dml_data["T"].isna().sum(),
        dml_data["Y"].isna().sum()
    ]
})

display(summary)

summary.to_csv(
    "day9_dml_dataset_summary.csv",
    index=False
)

print("Day 9 summary report saved.")

,Metric,Value
0,Rows before filtering,536641
1,Rows after filtering,524878
2,Treatment variable,UnitPrice
3,Outcome variable,Quantity
4,Number of covariates,3
5,Missing treatment values,0
6,Missing outcome values,0


Day 9 summary report saved.
